# Optimization workflow

The point of aggregation is to feed a smaller model and then map the answer back.
This page covers the handoff: what to give your model, how to **weight** columns,
how to **disaggregate** results, and how to **reuse** a clustering.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

In [ ]:
result = tsam.aggregate(data, n_clusters=6, period_duration="1D")

## What your model needs

Three objects describe the reduced problem:

- **`cluster_representatives`** — the typical periods (your model's time series input).
- **`cluster_counts`** — how many real periods each one stands for (its weight in the objective).
- **`cluster_assignments`** — which typical period each original period maps to.

In [ ]:
print("representatives:", result.cluster_representatives.shape)
print("counts:", dict(result.cluster_counts))
print("assignments (first 10):", result.cluster_assignments[:10])

## Weighting columns

When several columns are aggregated together, raise a column's **weight** to make
the clustering reproduce it more accurately — at the others' expense. Useful when
demand matters more than weather:

In [ ]:
weighted = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    weights={"GHI": 1.0, "T": 1.0, "Wind": 1.0, "Load": 20.0},
)
pd.DataFrame({"equal": result.accuracy.rmse, "Load x20": weighted.accuracy.rmse}).round(
    4
)

## Mapping results back

Say your model returns a dispatch decision *per typical period*. **Disaggregation**
expands it back onto the full timeline — one value for every original time step:

In [ ]:
# stand-in for an optimization output, indexed like the typical periods
model_output = result.cluster_representatives[["Load"]] * 0.5
full_year = result.disaggregate(model_output)
print(
    "typical-period output:", model_output.shape, "-> full timeline:", full_year.shape
)

## Reuse a clustering

Cluster once, then apply the **same** grouping to other data (e.g. a new scenario)
without re-running the clustering. It also serializes to JSON:

In [ ]:
clustering = result.clustering
applied = clustering.apply(data)  # same clustering, (re)applied
print("reapplied:", applied.cluster_representatives.shape)

clustering.to_json("clustering.json")  # persist...
from tsam import ClusteringResult

reloaded = ClusteringResult.from_json("clustering.json").apply(
    data
)  # ...and reuse later
print("from JSON:", reloaded.cluster_representatives.shape)